# 00 — Download AlleNoise
Downloads the AlleNoise benchmark (Allegro, Polish e-commerce; ~502k product titles, ~5.6k categories, ~15% real label noise, noisy + clean labels + category taxonomy).

Two sources are supported — pick one in the config cell:
- **github** (recommended): `git clone` + git-lfs of `allegro/AlleNoise`.
- **zenodo**: direct download via the Zenodo record API.

Paper: arXiv:2407.10992 (AISTATS 2025). Repo: https://github.com/allegro/AlleNoise

In [9]:
# Notebook: 00_download_data
# Shared plotting style: grayscale seaborn, dpi 600, PNG + PDF, no captions.
import os, numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
sns.set_theme(style="whitegrid", context="paper", font_scale=1.1)
plt.rcParams["axes.edgecolor"] = "0.2"; plt.rcParams["axes.linewidth"] = 0.8
plt.rcParams["font.family"] = "DejaVu Sans"
GREYS = ["#111111", "#555555", "#888888", "#bbbbbb", "#dddddd"]
FIG = os.path.join("..", "results", "figures"); TAB = os.path.join("..", "results", "tables")
os.makedirs(FIG, exist_ok=True); os.makedirs(TAB, exist_ok=True)
def savefig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIG, f"{name}.{ext}"), dpi=600, bbox_inches="tight")
    plt.close(fig)

# --- config ---
SOURCE = "github"          # "github" or "zenodo"
DATA_DIR = os.path.join("..", "data")
ZENODO_RECORD = "11500851" # AlleNoise Zenodo record id (see paper page for the current one)
os.makedirs(DATA_DIR, exist_ok=True)

# --- download ---
import shutil, subprocess, sys

def sh(cmd):
    print("$", " ".join(cmd)); subprocess.run(cmd, check=True)

if SOURCE == "github":
    repo = os.path.join(DATA_DIR, "_AlleNoise_repo")
    if not os.path.isdir(repo):
        # git-lfs must be installed once on your machine: `git lfs install`
        sh(["git", "lfs", "install"])
        sh(["git", "clone", "https://github.com/allegro/AlleNoise.git", repo])
    # copy the dataset files up into DATA_DIR
    src = os.path.join(repo, "allenoise")
    for fn in os.listdir(src):
        shutil.copy2(os.path.join(src, fn), os.path.join(DATA_DIR, fn))
    print("copied:", os.listdir(src))

elif SOURCE == "zenodo":
    import requests
    from tqdm import tqdm
    rec = requests.get(f"https://zenodo.org/api/records/{ZENODO_RECORD}", timeout=60).json()
    for f in rec["files"]:
        url = f["links"]["self"]; name = f["key"]
        dst = os.path.join(DATA_DIR, name)
        if os.path.exists(dst):
            print("exists:", name); continue
        print("downloading", name, f.get("size"))
        r = requests.get(url, stream=True, timeout=600); r.raise_for_status()
        with open(dst, "wb") as out:
            for chunk in tqdm(r.iter_content(chunk_size=1 << 20)):
                out.write(chunk)
print("files in DATA_DIR:", sorted(os.listdir(DATA_DIR)))

# --- load & auto-detect schema (robust: delimiter sniff, LFS-pointer check, raw head) ---
import csv
def find_csv(*keywords):
    for fn in os.listdir(DATA_DIR):
        low = fn.lower()
        if fn.endswith(".csv") and all(k in low for k in keywords):
            return os.path.join(DATA_DIR, fn)
    return None

full_path = find_csv("full") or find_csv("dataset") or find_csv("allenoise")
assert full_path, f"could not find the main CSV in {DATA_DIR}; files={os.listdir(DATA_DIR)}"
print("using:", full_path, "| size:", os.path.getsize(full_path), "bytes")

# 0) print the raw first lines so the true format is visible
with open(full_path, "r", encoding="utf-8", errors="replace") as fh:
    head_lines = [fh.readline() for _ in range(12)]
print("----- raw first lines -----")
for i, ln in enumerate(head_lines, 1):
    print(f"{i:2d}| {ln.rstrip()[:200]}")

# 1) git-lfs pointer => the data was not actually pulled
if head_lines and head_lines[0].startswith("version https://git-lfs"):
    raise RuntimeError("git-lfs POINTER, not data. In the downloaded repo run: "
                       "git lfs install && git lfs pull")

# 2) sniff the delimiter, then load with the tolerant python engine
sample = "".join(head_lines)
try:
    sep = csv.Sniffer().sniff(sample, delimiters=[",", "\t", ";", "|"]).delimiter
except Exception:
    sep = ","
print("sniffed delimiter:", repr(sep))
try:
    df = pd.read_csv(full_path, sep=sep, engine="python")
except Exception as e:
    print("strict parse failed:", e, "\n-> retrying with on_bad_lines='skip'")
    df = pd.read_csv(full_path, sep=sep, engine="python", on_bad_lines="skip")
print("loaded:", full_path, "shape:", df.shape)
print("columns:", list(df.columns))
print(df.head(3).to_string())

# Heuristic column detection (edit here if names differ).
def pick(cands):
    for c_ in df.columns:
        if any(k in c_.lower() for k in cands):
            return c_
    return None
COL_TEXT  = pick(["text", "title", "name"])
COL_NOISY = pick(["noisy"])
COL_CLEAN = pick(["clean", "true", "verified"])
print("\ndetected -> text:", COL_TEXT, "| noisy:", COL_NOISY, "| clean:", COL_CLEAN)
assert COL_TEXT and COL_NOISY and COL_CLEAN, "adjust the column picks above to match the schema"

# --- basic profile & save a normalized copy for the pipeline ---
noise_rate = float((df[COL_NOISY] != df[COL_CLEAN]).mean())
n_cat = int(pd.concat([df[COL_NOISY], df[COL_CLEAN]]).nunique())
print(f"rows           : {len(df):,}")
print(f"categories     : {n_cat:,}")
print(f"real noise rate: {noise_rate:.3%}  (fraction where noisy != clean)")

norm = df.rename(columns={COL_TEXT: "text", COL_NOISY: "noisy_category", COL_CLEAN: "clean_category"})
keep_cols = ([ "offer_id"] if "offer_id" in norm.columns else []) + ["text", "noisy_category", "clean_category"]
norm = norm[keep_cols].reset_index(drop=True)
norm.to_parquet(os.path.join(DATA_DIR, "allenoise_norm.parquet"))
print("saved ../data/allenoise_norm.parquet")

# taxonomy mapping (optional; used for group construction in nb 03) — robust loader
map_path = find_csv("map") or find_csv("category")
if map_path:
    with open(map_path, "r", encoding="utf-8", errors="replace") as fh:
        mhead = [fh.readline() for _ in range(8)]
    print("----- category_mapping raw first lines -----")
    for i, ln in enumerate(mhead, 1):
        print(f"{i:2d}| {ln.rstrip()[:200]}")
    try:
        msep = csv.Sniffer().sniff("".join(mhead), delimiters=[",", "\t", ";", "|"]).delimiter
    except Exception:
        msep = "\t"
    print("sniffed delimiter:", repr(msep))
    cmap = pd.read_csv(map_path, sep=msep, engine="python")
    cmap.to_parquet(os.path.join(DATA_DIR, "category_mapping.parquet"))
    print("\ncategory_mapping columns:", list(cmap.columns), "| shape:", cmap.shape)
    print(cmap.head(5).to_string())
else:
    print("no category_mapping csv found; nb 03 will fall back to label-id grouping")

copied: ['category_mapping.csv', 'data_sheet.pdf', 'full_dataset.csv', 'metadata.json']
files in DATA_DIR: ['.gitkeep', '_AlleNoise_repo', 'allenoise_norm.parquet', 'category_mapping.csv', 'data_sheet.pdf', 'full_dataset.csv', 'metadata.json', 'pi_memmap.npy']
using: ..\data\full_dataset.csv | size: 34294519 bytes
----- raw first lines -----
 1| offer_id	text	clean_category_id	noisy_category_id
 2| 11124013881	VU meter LED stereo audio control indicator	19	19
 3| 10005782428	UV meter stereo audio control indicators	19	19
 4| 10504298423	Taga Harmony PF-1000 v.2 black - power lines	19	19
 5| 10163057737	DC-Blocker - AC FILTER CONDITIONER 230V AUDIO	19	19
 6| 7127925653	Audio Receiver 30PIN bluetooth BOSE ONKYO iPhone	19	19
 7| 7247433032	Home Theater Antenna Radio Tower SAMSUNG 1,8m	19	19
 8| 10884184452	MONACOR HF-145 rubber feet	19	19
 9| 8044474906	Receiver CSR8675 Bluetooth 5.0 APTX HD + PCM5102	19	19
10| 10024497942	DAC / Bluetooth Receiver 5.0 APTX-HD LDAC CSR8675	19	19
11| 107814